# 外れ値検出(標準偏差 / z-score)

`formatted_trial.csv` の各数値列について z = (x - mean) / std を計算し、
|z| > しきい値 となる行を外れ値としてフラグします。

再現性のため、mean / std はサンプル標準偏差(ddof=1, pandasの既定)を使用し、乱数は使いません。

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 設定

In [3]:
###実験1

INPUT_PATH = "../data/formatted_trial_exp1.csv"   # 入力CSVのパス
THRESHOLD = 3.0                        # |z| > THRESHOLD を外れ値とする
OUTPUT_PATH = "../data/formatted_trial_flagged_exp1.csv"                    # None なら "<入力名>_flagged.csv" に自動設定

# 外れ値チェック対象の数値列(必要に応じて増減してください)
TARGET_COLUMNS = [
    "Page Submit_al",
    "Alertness Level",
    "Affect Level"
]

In [ ]:
###実験2

INPUT_PATH = "../data/formatted_trial_exp2.csv"   # 入力CSVのパス
THRESHOLD = 3.0                        # |z| > THRESHOLD を外れ値とする
OUTPUT_PATH = "../data/formatted_trial_flagged_exp2.csv"                    # None なら "<入力名>_flagged.csv" に自動設定

# 外れ値チェック対象の数値列(必要に応じて増減してください)
TARGET_COLUMNS = [
    "Page Submit_wl",
    "Page Submit_al",
    "Water Level",
    "Alertness Level",
    "Affect Level",
]

## データ読み込み

In [4]:
df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")

missing = [c for c in TARGET_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"以下の列がCSVに存在しません: {missing}")

print(df.shape)
df.head()

(9500, 9)


,responseId,frame_block,affect_trial,Page Submit_al,Alertness Level,Affect Level,frame_1,frame_2,Message Group
0,R_9nvFNaU5rMNP8Vi,0,0,5.553,28.0,39.0,8.803,半分しか入っていない,neutral
1,R_9nvFNaU5rMNP8Vi,0,1,13.783,45.0,39.0,8.803,半分しか入っていない,neutral
2,R_9nvFNaU5rMNP8Vi,0,2,4.873,38.0,49.0,8.803,半分しか入っていない,neutral
3,R_9nvFNaU5rMNP8Vi,0,3,14.086,82.0,17.0,8.803,半分しか入っていない,neutral
4,R_9nvFNaU5rMNP8Vi,0,4,13.431,60.0,48.0,8.803,半分しか入っていない,neutral


## z-score の計算

In [5]:
def compute_zscores(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """指定した列について z = (x - mean) / std を計算し、
    'z_<列名>' という列を追加した DataFrame を返す。"""
    out = df.copy()
    for col in columns:
        mean = df[col].mean()
        std = df[col].std()  # ddof=1 (サンプル標準偏差)
        out[f"z_{col}"] = (df[col] - mean) / std
    return out


df_z = compute_zscores(df, TARGET_COLUMNS)
df_z[[f"z_{c}" for c in TARGET_COLUMNS]].describe()

,z_Page Submit_al,z_Alertness Level,z_Affect Level
count,9.500000e+03,9.500000e+03,9.500000e+03
mean,-7.180221e-17,1.795055e-16,1.436044e-16
std,1.000000e+00,1.000000e+00,1.000000e+00
min,-6.933163e-01,-2.656756e+00,-2.471304e+00
25%,-2.984998e-01,-5.617834e-01,-5.373931e-01
50%,-1.171673e-01,-3.804029e-02,-5.391539e-02
75%,1.269836e-01,4.857028e-01,6.229533e-01
max,7.410925e+01,2.580675e+00,2.363473e+00


## 外れ値のフラグ付け

In [6]:
def flag_outliers(df_with_z: pd.DataFrame, columns: list[str], threshold: float) -> pd.DataFrame:
    """|z| > threshold となる行に is_outlier=True を付け、
    どの列が原因かを outlier_columns にまとめる。"""
    out = df_with_z.copy()
    z_cols = [f"z_{c}" for c in columns]

    is_outlier_per_col = out[z_cols].abs().gt(threshold)
    out["is_outlier"] = is_outlier_per_col.any(axis=1)
    out["outlier_columns"] = is_outlier_per_col.apply(
        lambda row: ", ".join(row.index[row].str.replace("z_", "", regex=False)),
        axis=1,
    )
    return out


df_flagged = flag_outliers(df_z, TARGET_COLUMNS, THRESHOLD)
df_flagged["is_outlier"].sum()

np.int64(35)

## サマリー

In [7]:
n_total = len(df_flagged)
print(f"閾値: |z| > {THRESHOLD}")
print(f"総行数: {n_total}\n")

print("=== 列ごとの外れ値件数 ===")
for col in TARGET_COLUMNS:
    n = (df_flagged[f"z_{col}"].abs() > THRESHOLD).sum()
    print(f"  {col:20s}: {n:5d} 件")

n_rows = int(df_flagged["is_outlier"].sum())
print(f"\n外れ値を1つ以上含む行: {n_rows} / {n_total} ({n_rows / n_total:.1%})")

閾値: |z| > 3.0
総行数: 9500

=== 列ごとの外れ値件数 ===
  Page Submit_al      :    35 件
  Alertness Level     :     0 件
  Affect Level        :     0 件

外れ値を1つ以上含む行: 35 / 9500 (0.4%)


In [8]:
if "responceId" in df_flagged.columns:
    by_resp = (
        df_flagged.groupby("responceId")["is_outlier"]
        .sum()
        .sort_values(ascending=False)
    )
    by_resp = by_resp[by_resp > 0]
    print(f"外れ値を1件以上持つ回答者: {len(by_resp)} 名\n")
    print("=== 回答者別 外れ値件数(上位10名) ===")
    print(by_resp.head(10).to_string())

## 外れ値の詳細確認

In [9]:
id_cols = [c for c in ["responceId", "frame_block", "ref_subblock"] if c in df_flagged.columns]
df_flagged.loc[df_flagged["is_outlier"], id_cols + TARGET_COLUMNS + ["outlier_columns"]]

,frame_block,Page Submit_al,Alertness Level,Affect Level,outlier_columns
1132,6,89.138,50.0,40.0,Page Submit_al
1203,0,70.232,45.0,45.0,Page Submit_al
1903,0,63.998,40.0,49.0,Page Submit_al
2465,13,189.903,38.0,60.0,Page Submit_al
3639,7,57.268,50.0,40.0,Page Submit_al
3707,1,55.508,50.0,50.0,Page Submit_al
4404,0,57.240,45.0,45.0,Page Submit_al
4503,0,331.071,50.0,40.0,Page Submit_al
4702,0,58.113,20.0,10.0,Page Submit_al
5403,0,66.518,55.0,40.0,Page Submit_al


## CSVへの書き出し

In [10]:
output_path = Path(OUTPUT_PATH) if OUTPUT_PATH else Path(INPUT_PATH).with_name(
    Path(INPUT_PATH).stem + "_flagged.csv"
)

keep_cols = list(df.columns) + ["is_outlier", "outlier_columns"]
df_flagged[keep_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"フラグ付きCSVを出力しました: {output_path}")

フラグ付きCSVを出力しました: ../data/formatted_trial_flagged_exp1.csv
